# X-AlignFact v2 — Colab driver
Runs the reproducible pipeline part by part. Every number in the revised manuscript comes from `runs/` and `results/` produced here.

Copy `xalignfact_v2.zip` to `MyDrive/` first. Runtime: A100 GPU.

## Part 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/xalignfact_v2'          # outputs (data, runs, results)
LEGACY_PAR = '/content/drive/MyDrive/xalignfact/data/parallel/parallel_en_vi.json'  # v1 translations, reused

!rm -rf /content/xalignfact_v2 && unzip -q /content/drive/MyDrive/xalignfact_v2.zip -d /content/
%cd /content/xalignfact_v2
!pip install -q -r requirements.txt

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python tests/smoke_test.py   # must end with SMOKE TEST PASSED

## Part 1 — Data (leakage-free splits, overlap report, tokenisation, BM25 test sets)
Check `data/prepare.log` for the detected claim field and dropped rows, and `data/overlap_report.json` (all train/dev/test overlaps must be 0).

In [ ]:
!python scripts/prepare_data.py --root $ROOT --legacy-parallel $LEGACY_PAR
!cat $ROOT/data/manifest.json | head -60

## Part 3 — Main model, 5 seeds (EXP-2)

In [ ]:
# seed 42 keeps its checkpoints for the Part 6 analyses; --group main then runs the other 4 seeds
!python scripts/run.py --root $ROOT --variants full --seeds 42 --save-best --save-phase-ckpts
!python scripts/run.py --root $ROOT --group main

## Part 5a — Baselines with the same pipeline (EXP-6)

In [ ]:
!python scripts/run.py --root $ROOT --group baselines
!pip install -q underthesea
!python scripts/baseline_phobert.py --root $ROOT

## Part 5b — Ablations (EXP-3)

In [ ]:
!python scripts/run.py --root $ROOT --group ablations
# optional: !python scripts/run.py --root $ROOT --group optional

## Part 5c — Aggregation: mean ± SD over seeds, paired bootstrap vs. `full`, Holm correction

In [ ]:
!python scripts/aggregate.py --root $ROOT
import pandas as pd
pd.read_csv(f'{ROOT}/results/comparisons_vs_full.csv').round(4)

## Part 6 — Analyses (EXP-4, EXP-7)
t-SNE on test, error counts, gradient cosines per phase (from the seed-42 checkpoints) and the translation annotation sheet.

In [ ]:
!python scripts/run_analysis.py --root $ROOT --seed 42

## Part 7 — Paper figures (no training)
Builds Fig. 3–7 and the appendix figures from existing outputs into `results/figures/`. Fig. 7 and the t-SNE need `runs_ood/{full,mt_xlmr_vi}/seed42/best.pt` (EXP-9 `--save-best`).

In [ ]:
!python scripts/make_figures.py --root $ROOT
!cd $ROOT/results && zip -qr /content/xalignfact_v2_figures.zip figures && ls -lh /content/xalignfact_v2_figures.zip

## Send to the writing step
Zip `results/` and every `runs/*/seed*/results.json` + `train_log.json` (predictions optional).

In [ ]:
!cd $ROOT && zip -qr /content/xalignfact_v2_results.zip results runs -i '*.json' '*.csv' '*.png' && ls -lh /content/xalignfact_v2_results.zip